In [4]:
#Imports
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import amuse
from amuse.lab import *
from amuse.ext.orbital_elements import new_binary_from_orbital_elements
from amuse.ext.orbital_elements import get_orbital_elements_from_binary
from amuse.plot import scatter
from amuse.community.ph4.interface import Ph4
from matplotlib.animation import FuncAnimation, FFMpegWriter


In [2]:
print('hi')

hi


In [ ]:

def semi_to_orbital_period(a, mass):
    return ((4*np.pi**2*a**3)/(constants.G*mass)).sqrt()

def initial_conditions():
    
    m_sun = 1 | units.Msun
    m_saturn = 0.00028 | units.Msun
    a_saturn = 9.5 | units.au
    e_saturn = 0.05
    
    #The constants.G only has to be defined if you do not work in SI units (which here we do but oh well)
    bodies = new_binary_from_orbital_elements(m_sun, m_saturn, a_saturn, e_saturn, G = constants.G)
    bodies.name = ['sun', 'saturn']
    
    m_japetus = 1.8e+21 | units.kg
    a_japetus = 3.5617e+6 | units.km
    e_japetus = 0.028
    
    jbinary = new_binary_from_orbital_elements(m_saturn, m_japetus, a_japetus, e_japetus)
    jbinary.name = ['fake_saturn', 'japetus']
    japetus = jbinary[jbinary.name=='japetus']
    jbinary.position -= jbinary[0].position
    jbinary.velocity -= jbinary[0].velocity
    
    saturn = bodies[bodies.name=='saturn']
    japetus.position += saturn.position
    japetus.velocity += saturn.velocity
    
#     print(japetus)
    
    bodies.add_particles(japetus)
    
    return bodies


if __name__ in ('__main__'):
    bodies = initial_conditions()
    a = [] |units.au
    e = []
    orbit = get_orbital_elements_from_binary(bodies[0:2], G=constants.G)
    print(f'saturns orbit {orbit}')
    a.append(orbit[2])
    e.append(orbit[3])
    orbit = get_orbital_elements_from_binary(bodies[1:3], G=constants.G)
    print(f'japetus orbit {orbit}')
    a.append(orbit[2])
    e.append(orbit[3])

    # scatter(bodies.x, bodies.y, s=2+300*np.sqrt(bodies.mass/bodies.mass.sum()), c='b')
#     plt.show()
    
#     scatter(a,e)
#     plt.show()
    
    converter = nbody_system.nbody_to_si(bodies.mass.sum(), 1 | units.au)
    
    gravity = Ph4(converter)
    gravity.particles.add_particles(bodies)
    print(f'Gravity particles: {gravity.particles}')
    print(f'bodies={bodies}')
    
    channel_from_gravity = gravity.particles.new_channel_to(bodies)
    channel_from_gravity.copy()
    print(f'bodies={bodies}')

    Porb = semi_to_orbital_period(a[0], bodies[0:2].mass.sum())
    Ek0 = bodies.kinetic_energy()
    Ep0 = bodies.potential_energy()
    Etot0 = Ek0 + Ep0

    time = 0 | units.yr
    dt = Porb/100
    index = 0

    while gravity.model_time<Porb:
        time += dt
        gravity.evolve_model(time)
        channel_from_gravity.copy()
        Ek = bodies.kinetic_energy()
        Ep = bodies.potential_energy()
        Etot = Ek + Ep
        print(f'time = {time.in_(units.yr)}, dE = {(Etot-Etot0)/Etot0}')
        # bodies.position()
        plt.xlim(-10,10)
        plt.ylim(-10,10)
        scatter(bodies.x, bodies.y, s=2+300*np.sqrt(bodies.mass/bodies.mass.sum()), c='r')
        index += 1
        filename = f'snapshot_i{index:04}.jpg'
        plt.savefig(filename)
        plt.close()
    gravity.stop()
    plt.show()
    

    

/Users/liam/miniforge3/envs/Amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


saturns orbit (quantity<1.0 1.98892e+30 * kg>, quantity<0.00028 1.98892e+30 * kg>, quantity<9.5 149597870691.0 * m>, 0.050000000000000579, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
japetus orbit (quantity<0.00028 1.98892e+30 * kg>, quantity<9.05013776321e-10 1.98892e+30 * kg>, quantity<0.0238084939548 149597870691.0 * m>, 0.02800000000006753, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
Gravity particles:                  key         mass  potential_in_code       radius     timestep           vx           vy           vz            x            y            z
                   -           kg  m**2 * s**-2            m            s    m * s**-1    m * s**-1    m * s**-1            m            m            m
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========
 4718270901310508905    1.989e+30   -2.753e+04   -1.496e+11    1.5

In [ ]:

# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation, FFMpegWriter

# from amuse.lab import *
# # Keep your existing imports if these are already defined:
# # constants, units, nbody_system, Ph4, etc.


if __name__ == '__main__':
    bodies = initial_conditions()

    a = [] | units.au
    e = []

    orbit = get_orbital_elements_from_binary(
        bodies[0:2], G=constants.G
    )
    print(f'Saturn orbit: {orbit}')
    a.append(orbit[2])
    e.append(orbit[3])

    orbit = get_orbital_elements_from_binary(
        bodies[1:3], G=constants.G
    )
    print(f'Japetus orbit: {orbit}')
    a.append(orbit[2])
    e.append(orbit[3])

    converter = nbody_system.nbody_to_si(
        bodies.mass.sum(), 1 | units.au
    )

    gravity = Ph4(converter)
    gravity.particles.add_particles(bodies)

    channel_from_gravity = gravity.particles.new_channel_to(bodies)
    channel_from_gravity.copy()

    Porb = semi_to_orbital_period(
        a[0], bodies[0:2].mass.sum()
    )

    Ek0 = bodies.kinetic_energy()
    Ep0 = bodies.potential_energy()
    Etot0 = Ek0 + Ep0

    dt = Porb / 100
    n_frames = 100

    # Set up the figure once
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.set_xlabel('x [AU]')
    ax.set_ylabel('y [AU]')
    ax.set_title('Saturn and Japetus orbital evolution')
    ax.grid(True, alpha=0.3)

    # Convert positions to AU for plotting
    x = bodies.x.value_in(units.au)
    y = bodies.y.value_in(units.au)

    sizes = 2 + 300 * np.sqrt(
        bodies.mass.value_in(units.Msun)
        / bodies.mass.sum().value_in(units.Msun)
    )

    points = ax.scatter(x, y, s=sizes, c='red')

    time_text = ax.text(
        0.02, 0.98, '',
        transform=ax.transAxes,
        va='top'
    )

    def init():
        time_text.set_text('')
        return points, time_text

    def update(frame):
        time = (frame + 1) * dt

        # Advance the AMUSE simulation
        gravity.evolve_model(time)

        # Copy updated positions from the gravity solver
        # back into the bodies particle set
        channel_from_gravity.copy()

        x = bodies.x.value_in(units.au)
        y = bodies.y.value_in(units.au)

        # Update the existing scatter plot
        points.set_offsets(np.column_stack((x, y)))

        # Update the elapsed simulation time
        time_text.set_text(
            f't = {time.value_in(units.yr):.3f} yr'
        )

        # Monitor energy conservation
        Ek = bodies.kinetic_energy()
        Ep = bodies.potential_energy()
        Etot = Ek + Ep
        dE = ((Etot - Etot0) / Etot0)

        # print(
        #     f'time = {time.value_in(units.yr):.5f} yr, '
        #     f'dE = {dE:.3e}'
        # )

        return points, time_text

    ani = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        init_func=init,
        interval=40,
        blit=True,
        repeat=False
    )

    # Export the animation to MP4
    writer = FFMpegWriter(fps=10, bitrate=1800)

    ani.save(
        'saturn_japetus.mp4',
        writer=writer,
        dpi=150
    )

    print('Animation saved to saturn_japetus.mp4')

    gravity.stop()
    plt.close(fig)


/Users/liam/miniforge3/envs/Amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


Saturn orbit: (quantity<1.0 1.98892e+30 * kg>, quantity<0.00028 1.98892e+30 * kg>, quantity<9.5 149597870691.0 * m>, 0.050000000000000579, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
Japetus orbit: (quantity<0.00028 1.98892e+30 * kg>, quantity<9.05013776321e-10 1.98892e+30 * kg>, quantity<0.0238084939548 149597870691.0 * m>, 0.02800000000006753, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
time = 0.29274 yr, dE = -2.184e-07
time = 0.58549 yr, dE = -2.185e-07
time = 0.87823 yr, dE = -4.295e-07
time = 1.17097 yr, dE = -6.825e-07
time = 1.46372 yr, dE = -6.824e-07
time = 1.75646 yr, dE = -8.529e-07
time = 2.04920 yr, dE = -1.119e-06
time = 2.34195 yr, dE = -1.328e-06
time = 2.63469 yr, dE = -1.447e-06
time = 2.92743 yr, dE = -1.688e-06
time = 3.22018 yr, dE = -1.945e-06
time = 3.51292 yr, dE = -2.009e-06
time = 3.80566 yr, dE = -2.138e-06
time = 4.09841 yr, dE = -2.137e-06
time = 4.39115 yr, dE = -2.144e-06
time = 4.68389 yr, 